<img src="../assets/logo-banner.png" alt="radar-datatree — Cloud-native, time-aware weather radar datasets" width="800">

---

# NEXRAD KLOT — Cloud-Native Access in 5 Lines

**Data source:** NEXRAD Level II radar archive on the [AWS Open Data Registry](https://registry.opendata.aws/nexrad-arco/), curated as an ARCO Radar DataTree at `s3://nexrad-arco`.

This dataset is a hierarchical, time-indexed view of the NEXRAD KLOT (Chicago, Illinois) radar archive — radar moments organized by [Volume Coverage Pattern](https://www.weather.gov/jetstream/vcp_max) (VCP) and sweep, written as Zarr v3 with [Icechunk](https://icechunk.io/) transactional metadata.

---

## Why a DataTree?

- **Hierarchical** — every VCP × sweep combination is a node you can navigate (`dt["VCP-12/sweep_0"]`), not a separate file you have to download and parse.
- **Time-indexed** — every sweep stacks all of its scans along a `vcp_time` dimension, so `.sel(vcp_time=...)` replaces the file-iteration loop.
- **Lazy** — opening the full archive fetches metadata only (~MB). Variables stream from S3 on demand when you reduce or plot them.

For the architecture and the scaling benchmarks, see [Ladino-Rincón & Nesbitt (2025)](https://doi.org/10.48550/arXiv.2510.24943).

---

In [ ]:
import icechunk

storage = icechunk.s3_storage(
    bucket="nexrad-arco",
    prefix="KLOT",
    region="us-east-1",
    anonymous=True,
)
session = icechunk.Repository.open(storage).readonly_session("main")

In [ ]:
import xarray as xr
import xradar  # noqa: F401  — registers the .xradar accessor

dt = xr.open_datatree(session.store, engine="rustytree", chunks=None)
dt

Opened with `engine="rustytree"` — a Rust-backed xarray DataTree backend that walks the Zarr v3 hierarchy concurrently across one FFI crossing. Drop-in replacement for `engine="zarr"`, ~10× faster on icechunk repos served from object storage. See [`rustytree-xarray` on PyPI](https://pypi.org/project/rustytree-xarray/).

---

In [ ]:
import cmweather  # noqa: F401  — registers ChaseSpectral colormap
import matplotlib.pyplot as plt

scan = (
    dt["VCP-12/sweep_0"]
    .to_dataset(inherit="all_coords")
    .xradar.georeference()
    .sel(vcp_time="2021-08-09", method="nearest")
)

fig, ax = plt.subplots(figsize=(7, 6))
scan.DBZH.plot(x="x", y="y", cmap="ChaseSpectral", vmin=-10, vmax=70, ax=ax)
ax.set_title(f"DBZH — {str(scan.vcp_time.values)[:19]} UTC")
ax.set_aspect("equal")
plt.show()

## "Sweep_0 from every VCP"

The canonical multi-VCP query: get the lowest-elevation sweep across every Volume Coverage Pattern in the archive, in one call. The DataTree is filtered to only those nodes (with their VCP container groups auto-included as ancestors).

---

In [ ]:
sweeps_0 = xr.open_datatree(session.store, engine="rustytree", group="/*/sweep_0")
sweeps_0

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
panels = [
    ("DBZH", "ChaseSpectral", -10, 70, "Reflectivity [dBZ]"),
    ("ZDR", "ChaseSpectral", -2, 6, "Differential Reflectivity [dB]"),
    ("RHOHV", "Carbone11", 0.7, 1.0, "Cross-Correlation Coefficient"),
    ("PHIDP", "PD17", 0, 180, "Differential Phase [deg]"),
]
for ax, (var, cmap, vmin, vmax, label) in zip(axes.flat, panels, strict=True):
    scan[var].plot(
        ax=ax,
        x="x",
        y="y",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        cbar_kwargs={"label": label, "shrink": 0.8},
    )
    ax.set_title(var)
    ax.set_aspect("equal")
fig.suptitle(
    f"KLOT polarimetric snapshot — {str(scan.vcp_time.values)[:19]} UTC",
    fontsize=12,
)
plt.tight_layout()
plt.show()

## Where to next

- **[2. QVP Workflow Comparison](2.QVP-Workflow-Comparison)** — reproduce Ryzhkov et al. (2016) and benchmark file-based vs ARCO data access.
- **[AWS Open Data Registry — nexrad-arco](https://registry.opendata.aws/nexrad-arco/)** — dataset metadata, terms of use, and other bucket prefixes.
- **[rustytree-xarray](https://github.com/aladinor/rustytree)** — the Rust DataTree backend.

---

*Cite this work:* Ladino-Rincón, A., & Nesbitt, S. W. (2025). *Radar DataTree: A FAIR and Cloud-Native Framework for Scalable Weather Radar Archives.* arXiv:2510.24943. [doi:10.48550/arXiv.2510.24943](https://doi.org/10.48550/arXiv.2510.24943)